# 🎯 [K14_M07] 그래프 시각화 요청하기
### 학생 실습 워크북 & 프롬프트 설계 가이드 (Google Colab / Jupyter)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/K14/k14_m07_student.ipynb)

---

## 📌 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 / 과제** | 병영생활·부대관리 (Level 1) / K14 군 생활 데이터 분석 |
| **미션** | 미션 07. 그래프 시각화 요청하기 (소요시간: 약 1.5시간) |
| **실습 유형** | 군 생활 데이터 분석, HITL 수기 검산, 시각화 왜곡 방지 워크북 |

> **⚠️ [교육용 가상 자료 원칙]**
> 본 실습의 모든 자료는 안전 규정에 따라 **교육용 가상 부대 자료(가상 한빛대대)**로 구성되어 있습니다.
> 실제 군사 기밀이나 부대 세부 인적·작전 정보는 외부 AI 도구에 절대 입력하지 않습니다.



In [ ]:
# ============================================================================
# [Colab 환경 설정 및 헬퍼 함수] 데이터 분석, 가상 데이터 로더 & 마크다운 렌더러
# ============================================================================
!pip install -q pandas numpy tabulate matplotlib seaborn

import pandas as pd
import numpy as np
import json
import os
import re
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

# 폰트 설정 (Colab / Linux 호환)
plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial']
plt.rcParams['axes.unicode_minus'] = False

# ----------------------------------------------------------------------------
# [데이터셋 자동 로드] 가상 한빛대대 4주간 생활관 생활 데이터 (12행)
# ----------------------------------------------------------------------------
data_records = [
    {'주차': '1주차', '중대': '1중대', '인원(명)': 48, '체력단련실': 112, '싸지방': 54, 'PX': 140, '수면만족': 3.8, '식단만족': 3.9, '건의': 3},
    {'주차': '1주차', '중대': '2중대', '인원(명)': 52, '체력단련실': 98, '싸지방': 31, 'PX': 160, '수면만족': 3.6, '식단만족': 3.5, '건의': 4},
    {'주차': '1주차', '중대': '3중대', '인원(명)': 45, '체력단련실': 104, '싸지방': 47, 'PX': 118, '수면만족': 3.7, '식단만족': 3.2, '건의': 5},
    {'주차': '2주차', '중대': '1중대', '인원(명)': 48, '체력단련실': 118, '싸지방': 58, 'PX': 138, '수면만족': 3.8, '식단만족': 3.8, '건의': 2},
    {'주차': '2주차', '중대': '2중대', '인원(명)': 52, '체력단련실': 92, '싸지방': 29, 'PX': 166, '수면만족': 3.4, '식단만족': 3.4, '건의': 5},
    {'주차': '2주차', '중대': '3중대', '인원(명)': 45, '체력단련실': 106, '싸지방': 50, 'PX': 121, '수면만족': 3.6, '식단만족': 3.1, '건의': 7},
    {'주차': '3주차', '중대': '1중대', '인원(명)': 48, '체력단련실': 121, '싸지방': 61, 'PX': 142, '수면만족': 3.7, '식단만족': 4.0, '건의': 4},
    {'주차': '3주차', '중대': '2중대', '인원(명)': 52, '체력단련실': 85, '싸지방': 26, 'PX': 171, '수면만족': 3.2, '식단만족': 3.4, '건의': 7},
    {'주차': '3주차', '중대': '3중대', '인원(명)': 45, '체력단련실': 103, '싸지방': 49, 'PX': 117, '수면만족': 3.7, '식단만족': 3.3, '건의': 9},
    {'주차': '4주차', '중대': '1중대', '인원(명)': 48, '체력단련실': 125, '싸지방': 66, 'PX': 135, '수면만족': 3.7, '식단만족': 4.1, '건의': 3},
    {'주차': '4주차', '중대': '2중대', '인원(명)': 52, '체력단련실': 80, '싸지방': 24, 'PX': 178, '수면만족': 3.0, '식단만족': 3.3, '건의': 8},
    {'주차': '4주차', '중대': '3중대', '인원(명)': 45, '체력단련실': 108, '싸지방': 52, 'PX': 120, '수면만족': 3.7, '식단만족': 3.2, '건의': 12}
]

df = pd.DataFrame(data_records)

def load_dataset(mission_num):
    """미션 번호(1~10)에 해당하는 가상 데이터셋 JSON을 로드합니다."""
    filename = f"k14_m{mission_num:02d}_dataset.json"
    for base in [".", "uploads/K14", "../uploads/K14"]:
        filepath = os.path.join(base, filename)
        if os.path.exists(filepath):
            with open(filepath, "r", encoding="utf-8") as f:
                return json.load(f)
    print(f"⚠️ 데이터셋 파일을 찾을 수 없습니다: {filename}")
    return None

def show_result(mission_title, prompt, result):
    """학습자가 입력한 프롬프트와 분석 결과 및 검증 리포트를 시각적으로 렌더링합니다."""
    p_text = prompt.strip() if prompt.strip() else "(작성된 프롬프트가 없습니다)"
    r_text = result.strip() if result.strip() else "(기록된 결과가 없습니다)"
    
    md_content = f"""
---
### 📌 [{mission_title}] 실습 기록 리포트

#### 💬 입력 프롬프트
```text
{p_text}
```

#### 📋 분석 결과 및 팩트체크 검증 내역
{r_text}
---
"""
    display(Markdown(md_content))

print("✅ [K14] 실습 환경, 12행 생활관 데이터셋(df) 및 시각화 렌더러 준비 완료!")
display(df)



---

## 🎯 미션 07. 그래프 시각화 요청하기
**소요시간**: 약 1.5시간

### 📋 미션 배경 및 지침
#### 미션 배경 및 당위성

실제 부대 실무 현장에서는 불명확한 정보 전달이나 수동 업무 처리로 인해 불필요한 행정 소요가 발생하고 주요 공지·규정 확인이 지연되는 문제가 발생하기 쉽습니다. 따라서 **그래프 시각화를 요청하고 오해를 막습니다.** 과업을 체계적으로 수행하여 현장 업무 효율성을 극대화하고 정보 누락으로 인한 혼선을 사전에 방지해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **추세에는 선 그래프, 중대 비교에는 막대 그래프처럼 목적에 맞는 그래프를 AI에게 요청 수행하고 검증하는** 실습을 단계별로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동을 순차적으로 진행하며 핵심 산출물을 도출하고 검증 기준을 달성합니다.

**이번 미션 결과물:** `그래프 요청 프롬프트 2개, 그래프 점검표` 완성

### 📑 실습 서식 및 대조 점검 양식
미션 7 그래프 요청 프롬프트 틀과 점검표

| 위 [데이터]로 (선 그래프 / 막대 그래프)를 만들어 줘.
- 그릴 항목: 
- 가로축: 
- 세로축 이름과 단위: 
- 세로축은 0부터 시작하게 하거나 시작값을 명시해 줘.
- 제목과 범례를 넣어 줘.
그래프를 그릴 수 없으면 표 계산 도구에서 만드는 방법을 단계별로 알려 줘. |
| --- |

| **점검 항목** | **그래프 1** | **그래프 2** |
| --- | --- | --- |
| 축 이름과 단위가 맞다 | ( ) | ( ) |
| 축 시작값이 과장되지 않았다 | ( ) | ( ) |
| 값이 표와 일치한다 | ( ) | ( ) |
| 제목이 내용을 설명한다 | ( ) | ( ) |

### 🔍 평가 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가 기준:** [필수] 시각화 요청·검증 — 목적에 맞는 그래프를 요청하고 축과 단위를 데이터와 대조해 점검했는가? (PASS / FAIL)

#### 미션 PASS 조건

- 요구 산출물(그래프 요청 프롬프트 2개, 그래프 점검표)의 완성도 충족


- 원문 대조 점검을 통한 사실 검증 및 정확성 확보


- 교육용 가상 데이터 활용 및 군사보안 수칙 엄수


#### 채점자 판정 메모

> 단순 결과물 생성 여부뿐만 아니라 AI 결과물의 오류를 직접 찾아내고 바로잡았는지 대조표를 확인한다.



In [ ]:
# [미션 07.] 학생 실습 - 프롬프트 작성 및 결과 기록
mission_07_prompt = """
# [TODO] 추세 선 그래프(수면만족)와 비교 막대 차트(1인당 체련실) 생성 프롬프트 2종을 작성하세요.
# 예시: 축 이름, 단위, Y축 시작값(0부터 시작) 명시 및 왜곡 점검 체크리스트
"""

mission_07_result = """
# [TODO] AI 질의 응답 결과 및 실습 서식에 따른 분석/검산 기록을 여기에 작성하세요.
"""

# 렌더러 실행 (작성한 프롬프트와 분석 결과를 렌더링합니다)
show_result("미션 07. 그래프 시각화 요청하기", mission_07_prompt, mission_07_result)



In [ ]:
# [미션 07 실습 보조] AI 생성 시각화 코드 실행 및 왜곡(축/스케일) 점검
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# 1. 추세 선 그래프 (수면 만족도)
for comp in ['1중대', '2중대', '3중대']:
    sub = df[df['중대'] == comp]
    axes[0].plot(sub['주차'], sub['수면만족'], marker='o', linewidth=2, label=comp)
axes[0].set_title('주차별 수면 만족도 변화 추이 (Line Chart)', fontsize=13)
axes[0].set_ylim(0, 5)  # 5점 척도 전체 표시 (왜곡 방지)
axes[0].set_ylabel('만족도 점수 (1~5점)')
axes[0].grid(True, linestyle='--', alpha=0.5)
axes[0].legend()

# 2. 비교 막대 그래프 (1인당 체력단련실 이용)
comp_names = ['1중대', '2중대', '3중대']
personnel = df.groupby('중대')['인원(명)'].first()
gym_per_capita = df.groupby('중대')['체력단련실'].sum().div(personnel).loc[comp_names]
bars = axes[1].bar(comp_names, gym_per_capita, color=['#3366cc', '#dc3912', '#109618'], width=0.5)
axes[1].set_title('중대별 1인당 4주 체력단련실 이용 횟수 (Bar Chart)', fontsize=13)
axes[1].set_ylim(0, 12)  # Y축 0부터 시작 (정직한 시각화)
axes[1].set_ylabel('1인당 평균 이용 횟수(회)')
axes[1].grid(axis='y', linestyle='--', alpha=0.5)

# 막대 상단 수치 표기
for bar in bars:
    h = bar.get_height()
    axes[1].annotate(f'{h:.2f}회', xy=(bar.get_x() + bar.get_width()/2, h),
                     xytext=(0, 3), textcoords='offset points', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()